# ROGII - Wellbore Geology Prediction

**Reference:** 
- [ROGII-Wellbore-Geology-Prediction](https://www.kaggle.com/code/vishwasmishra1234/rogii-wellbore-geology-prediction)
- [XGB Starter - [CV 15]](https://www.kaggle.com/code/cdeotte/xgb-starter-cv-15)

# 1. Imports and configs

In [1]:
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor
from koolbox import Trainer
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
class CFG:
    data_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    log_path = Path("/kaggle/working/logs")
    
    seed = 42
    n_splits = 7
    cv = GroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

# 2. Data loading and preprocessing

In [3]:
def recent_mean_diff(values, window):
    values = values[-(window + 1):]
    if len(values) < 2:
        return 0.0
    
    return float(np.diff(values).mean())


def recent_slope(y_values, x_values, window):
    y_values = y_values[-window:]
    x_values = x_values[-window:]
    if len(y_values) < 2:
        return 0.0
    
    centered_x = x_values - x_values.mean()
    denominator = float(np.dot(centered_x, centered_x))
    if denominator == 0.0:
        return 0.0
    
    return float(np.dot(centered_x, y_values - y_values.mean()) / denominator)


def nearest_index(sorted_values, target):
    idx = int(np.searchsorted(sorted_values, target, side='left'))
    if idx >= len(sorted_values):
        return len(sorted_values) - 1
    
    if idx > 0 and abs(sorted_values[idx - 1] - target) <= abs(sorted_values[idx] - target):
        return idx - 1
    
    return idx

 
def fill_and_smooth_gr(values, fallback, radius):
    series = pd.Series(values, dtype='float32').interpolate(limit_direction='both').fillna(fallback)
    if radius <= 0:
        return series.to_numpy(dtype=np.float32)
    
    return series.rolling(radius * 2 + 1, center=True, min_periods=1).mean().to_numpy(dtype=np.float32)


def beam_predict(gr_values, tw_tvt, tw_gr, start_tvt, beam_size, move_cost, emit_scale, radius):
    start_idx = nearest_index(tw_tvt, start_tvt)
    smoothed_gr = fill_and_smooth_gr(gr_values, float(np.nanmean(tw_gr)), radius)

    states = {start_idx: 0.0}
    backpointers = []

    for gr_value in smoothed_gr:
        candidates = {}
        parents = {}
        for idx, cost in states.items():
            for delta in (-1, 0, 1):
                next_idx = idx + delta
                if next_idx < 0 or next_idx >= len(tw_tvt):
                    continue
                
                emit_cost = ((gr_value - tw_gr[next_idx]) ** 2) / emit_scale
                total_cost = cost + emit_cost + move_cost * abs(delta)
                previous = candidates.get(next_idx)
                
                if previous is None or total_cost < previous:
                    candidates[next_idx] = total_cost
                    parents[next_idx] = idx
        
        kept = sorted(candidates.items(), key=lambda item: item[1])[:beam_size]
        states = {idx: cost for idx, cost in kept}
        backpointers.append({idx: parents[idx] for idx, _ in kept})

    final_idx = min(states, key=states.get)
    path = [final_idx]
    for step in range(len(backpointers) - 1, 0, -1):
        path.append(backpointers[step][path[-1]])
    
    path.reverse()
    
    return tw_tvt[np.asarray(path, dtype=np.int32)]


def build_hidden_features(horizontal_path, typewell_path, is_train):
    well = horizontal_path.name.split('__')[0]
    df = pd.read_csv(horizontal_path)
    mask = df['TVT_input'].isna().to_numpy()
    if not mask.any():
        return None

    mask_start = int(np.flatnonzero(mask)[0])
    if mask_start == 0:
        return None

    known = df.iloc[:mask_start].copy()
    hidden = df.iloc[mask_start:].copy()
    last_known = known.iloc[-1]

    tw = pd.read_csv(typewell_path)
    tw_tvt = tw['TVT'].to_numpy(dtype=np.float32)
    tw_gr = tw['GR'].to_numpy(dtype=np.float32)

    gr_full = df['GR'].interpolate(limit_direction='both')
    if gr_full.isna().any():
        gr_full = gr_full.fillna(float(np.nanmean(tw_gr)))

    gr_roll5 = gr_full.rolling(5, center=True, min_periods=1).mean()
    gr_roll21 = gr_full.rolling(21, center=True, min_periods=1).mean()
    gr_roll50 = gr_full.rolling(50, center=True, min_periods=1).mean()
    gr_grad = gr_full.diff().fillna(0.0)

    gr_std5 = gr_full.rolling(5, center=True, min_periods=1).std().fillna(0.0)
    gr_std21 = gr_full.rolling(21, center=True, min_periods=1).std().fillna(0.0)
    gr_lag1 = gr_full.shift(1).bfill()
    gr_lead1 = gr_full.shift(-1).ffill()
    gr_lag5 = gr_full.shift(5).bfill()
    gr_lead5 = gr_full.shift(-5).ffill()
    gr_cumsum = gr_full.cumsum()

    known_tvt = known['TVT_input'].to_numpy(dtype=np.float32)
    known_md = known['MD'].to_numpy(dtype=np.float32)
    known_z = known['Z'].to_numpy(dtype=np.float32)

    prefix_typewell_gr = np.interp(known_tvt, tw_tvt, tw_gr)
    prefix_gr = gr_full.iloc[:mask_start].to_numpy(dtype=np.float32)
    prefix_residual = prefix_gr - prefix_typewell_gr
    prefix_tw_rmse = float(np.sqrt(np.mean(prefix_residual ** 2)))
    prefix_tw_mae = float(np.mean(np.abs(prefix_residual)))

    last_known_tvt = float(last_known['TVT_input'])
    hidden_gr = hidden['GR'].to_numpy(dtype=np.float32)
    beam_cons = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 30, 20.0, 144.0, 2)
    beam_loose = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 30, 8.0, 64.0, 2)

    hidden_gr_filled = gr_full.iloc[mask_start:].to_numpy(dtype=np.float32)
    offset_diffs = {
        f'tw_diff_{int(offset)}': hidden_gr_filled - np.float32(np.interp(last_known_tvt + float(offset), tw_tvt, tw_gr))
        for offset in np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], dtype=np.float32)
    }

    features = pd.DataFrame({
        'well': well,
        'prediction_id': [f'{well}_{row_idx}' for row_idx in hidden.index],
        'row_idx': hidden.index.to_numpy(dtype=np.int32),
        'last_known_tvt': np.float32(last_known_tvt),
        'known_len': np.int32(mask_start),
        'hidden_len': np.int32(len(hidden)),
        'frac_hidden': ((hidden.index - mask_start) / max(len(hidden) - 1, 1)).astype(np.float32),
        'md': hidden['MD'].to_numpy(dtype=np.float32),
        'z': hidden['Z'].to_numpy(dtype=np.float32),
        'x': hidden['X'].to_numpy(dtype=np.float32),
        'y': hidden['Y'].to_numpy(dtype=np.float32),
        'gr': hidden_gr_filled,
        'gr_missing': hidden['GR'].isna().to_numpy(dtype=np.int8),
        'gr_roll5': gr_roll5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_roll21': gr_roll21.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_roll50': gr_roll50.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_grad': gr_grad.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_std5': gr_std5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_std21': gr_std21.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lag1': gr_lag1.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lead1': gr_lead1.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lag5': gr_lag5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lead5': gr_lead5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_cumsum': (gr_cumsum.iloc[mask_start:] - gr_cumsum.iloc[mask_start - 1]).to_numpy(dtype=np.float32) if mask_start > 0 else gr_cumsum.iloc[mask_start:].to_numpy(dtype=np.float32),
        'dmd': (hidden['MD'] - float(last_known['MD'])).to_numpy(dtype=np.float32),
        'dz': (hidden['Z'] - float(last_known['Z'])).to_numpy(dtype=np.float32),
        'dx': (hidden['X'] - float(last_known['X'])).to_numpy(dtype=np.float32),
        'dy': (hidden['Y'] - float(last_known['Y'])).to_numpy(dtype=np.float32),
        'dx_dmd': ((hidden['X'] - float(last_known['X'])) / np.maximum(hidden['MD'] - float(last_known['MD']), 1e-5)).to_numpy(dtype=np.float32),
        'dy_dmd': ((hidden['Y'] - float(last_known['Y'])) / np.maximum(hidden['MD'] - float(last_known['MD']), 1e-5)).to_numpy(dtype=np.float32),
        'dz_dmd': ((hidden['Z'] - float(last_known['Z'])) / np.maximum(hidden['MD'] - float(last_known['MD']), 1e-5)).to_numpy(dtype=np.float32),
        'dist_xy': np.sqrt((hidden['X'] - float(last_known['X'])) ** 2 + (hidden['Y'] - float(last_known['Y'])) ** 2).to_numpy(dtype=np.float32),
        'dist_xyz': np.sqrt((hidden['X'] - float(last_known['X'])) ** 2 + (hidden['Y'] - float(last_known['Y'])) ** 2 + (hidden['Z'] - float(last_known['Z'])) ** 2).to_numpy(dtype=np.float32),
        'prefix_tvt_step20': np.float32(recent_mean_diff(known_tvt, 20)),
        'prefix_tvt_step100': np.float32(recent_mean_diff(known_tvt, 100)),
        'prefix_tvt_md_slope100': np.float32(recent_slope(known_tvt, known_md, 100)),
        'prefix_tvt_z_slope100': np.float32(recent_slope(known_tvt, known_z, 100)),
        'prefix_tw_rmse': np.float32(prefix_tw_rmse),
        'prefix_tw_mae': np.float32(prefix_tw_mae),
        'beam_cons_delta': (beam_cons - np.float32(last_known_tvt)).astype(np.float32),
        'beam_loose_delta': (beam_loose - np.float32(last_known_tvt)).astype(np.float32),
        'beam_gap': (beam_loose - beam_cons).astype(np.float32),
    })

    for name, values in offset_diffs.items():
        features[name] = values.astype(np.float32)

    if is_train:
        features['target'] = (hidden['TVT'].to_numpy(dtype=np.float32) - np.float32(last_known_tvt)).astype(np.float32)

    return features


def build_dataset(path, is_train):
    parts = []
    
    paths = sorted(path.glob('*__horizontal_well.csv'))
    for horizontal_path in tqdm(paths, desc=f"Building {path.name}"):
        well = horizontal_path.name.split('__')[0]
        typewell_path = path / f'{well}__typewell.csv'
        if not typewell_path.exists():
            continue
        
        features = build_hidden_features(horizontal_path, typewell_path, is_train=is_train)
        if features is not None:
            parts.append(features)

    return pd.concat(parts, ignore_index=True)

In [4]:
train_df = build_dataset(CFG.data_path / 'train', is_train=True)
test_df = build_dataset(CFG.data_path / 'test', is_train=False)

features = [column for column in train_df.columns if column not in {'well', 'prediction_id', 'target'}]

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"#features:   {len(features)}")

Building train:   0%|          | 0/773 [00:00<?, ?it/s]

Building test:   0%|          | 0/3 [00:00<?, ?it/s]

Train shape: (3783989, 54)
Test shape:  (14151, 53)
#features:   51


# 3. Training

In [5]:
model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective='regression',
    random_state=CFG.seed,
    force_row_wise=True,
    device="gpu",
    verbosity=-1,
    n_jobs=-1,
)

trainer = Trainer(
    estimator=model,
    cv=CFG.cv,
    metric=root_mean_squared_error,
    task="regression",
    verbose=True,
    cv_args={"groups": train_df["well"]},
    save=True,
    save_path=CFG.log_path
)

trainer.fit(
    train_df[features], 
    train_df["target"]
)

Training LGBMRegressor



1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


--- Fold 0 - root_mean_squared_error: 14.5445 - Time: 234.24 s
--- Fold 1 - root_mean_squared_error: 14.5313 - Time: 228.03 s
--- Fold 2 - root_mean_squared_error: 13.1075 - Time: 229.28 s
--- Fold 3 - root_mean_squared_error: 11.8626 - Time: 228.92 s
--- Fold 4 - root_mean_squared_error: 14.0281 - Time: 225.54 s
--- Fold 5 - root_mean_squared_error: 11.4267 - Time: 227.32 s
--- Fold 6 - root_mean_squared_error: 13.3042 - Time: 228.24 s

------ Overall root_mean_squared_error: 13.3256 - Mean root_mean_squared_error: 13.2578 ± 1.1461 - Time: 1651.31 s


# 4. Inference

In [6]:
test_preds = trainer.predict(test_df[features])
test_preds = test_preds + test_df['last_known_tvt']

In [7]:
submission = pd.DataFrame({
    'id': test_df['prediction_id'],
    'tvt': test_preds,
})

submission.to_csv("submission.csv", index=False)
submission

,id,tvt
0,000d7d20_1442,11747.859817
1,000d7d20_1443,11747.867227
2,000d7d20_1444,11747.894631
3,000d7d20_1445,11747.356012
4,000d7d20_1446,11747.349334
...,...,...
14146,00e12e8b_6379,11598.858156
14147,00e12e8b_6380,11598.864429
14148,00e12e8b_6381,11598.857002
14149,00e12e8b_6382,11598.912012
